In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import lingam
import networkx as nx
import matplotlib.pyplot as plt

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_LiM"
output_dir.mkdir(parents=True,exist_ok=True)

In [3]:
import time
"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    start = time.time()
    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)
    end=time.time()
    
    print(f"LiM took {(end - start)/60:.2f} minutes")
    return adj, cols

In [4]:
def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [5]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [6]:
def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [7]:
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)

df_enc = encode_mixed_df(df)
        
# Run LiM on this dataset
adj,nodes = run_lim(df_enc, seed=1)

#save recovered adjacency matrix to a csv
adj_df = pd.DataFrame(adj, index=df.columns, columns=df.columns)
adj_out_path = output_dir / "NIJ_graph_LiM_adj.csv"
adj_df.to_csv(adj_out_path, index=True)

out_path = output_dir / "NIJ_graph_LiM"

#draw graph with graphviz
draw_graphviz_dag(adj, out_path, nodes)

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.24050596 0.
  0.         0.         0.         0.         3.14200359 0.
  0.         1.59530689 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.74568787 0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.   

In [8]:
G = nx.DiGraph()
#add nodes
G.add_nodes_from(nodes)

#add directed edges, adj[i,j] = 1 => i -> j
for i, src in enumerate(nodes):
    for j, tgt in enumerate(nodes):
        if adj[i, j] == 1:
            G.add_edge(src, tgt)

In [9]:
import networkx as nx
from collections import Counter

AGE_PREFIX = "Age_at_Release_"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))


Original nodes: 21
After age-merge: 15
Markov blanket nodes: 2


In [10]:
node_labels = list(G_mb.nodes())          
adj = nx.to_numpy_array(G_mb, nodelist=node_labels, dtype=int)

out_path = output_dir/"NIJ_graph_LiM_PRUNED"
draw_graphviz_dag(adj, out_path, node_labels=node_labels, engine="dot")

In [11]:
print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mb.edges()))


Original edges: 13
After age-merge: 13
Markov blanket edges: 1
